# Group factor analysis (GFA) demo

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import latents.gfa.analysis as gfa_stats
import latents.gfa.simulation as gfa_sim
from latents.gfa import GFAFitConfig, GFAModel
from latents.gfa.inference import compute_lower_bound
from latents.observation import ObsParamsHyperPriorStructured, ObsParamsPosterior
from latents.plotting import (
    hinton_diagram,
    plot_dimensionalities,
    plot_dims_pairs,
    plot_var_exp,
    plot_var_exp_pairs,
)

## Generate data from the GFA model

In [ ]:
# Set a random seed, for reproducibility
random_seed = 0  # Set to None for no seeding
rng = np.random.default_rng(random_seed)

# Dataset characteristics
n_samples = 100  # Total number of samples
y_dims = np.array([10, 10, 10])  # Dimensionality of each observed group
n_groups = len(y_dims)  # Total number of groups
x_dim = 7  # Latent dimensionality
snr = 1.0 * np.ones(n_groups)  # Signal-to-noise ratio of each group

# Build up the desired sparsity pattern of the loading matrices, a
# (n_groups x x_dim) array. Row i corresponds to group i. Column j
# corresponds to latent j. A value of np.inf indicates that a latent is
# NOT present in a group. The corresponding loadings will be 0 for that
# group. The remaining hyperparameters are not very important, and can
# be left alone.
sparsity_pattern = np.array(
    [
        [1, 1, 1, np.inf, 1, np.inf, np.inf],
        [1, 1, np.inf, 1, np.inf, 1, np.inf],
        [1, np.inf, 1, 1, np.inf, np.inf, 1],
    ],
)
MAG = 100  # Control the variance of alpha parameters (larger = less var.)
sim_priors = ObsParamsHyperPriorStructured(
    a_alpha=MAG * sparsity_pattern,
    b_alpha=MAG * np.ones_like(sparsity_pattern),
    a_phi=1.0,
    b_phi=1.0,
    beta_d=1.0,
)

# Simulate data
Y, X_true, obs_params_true = gfa_sim.simulate(
    n_samples,
    y_dims,
    x_dim,
    sim_priors,
    snr,
    rng=rng,
)

## Fit a GFA model to data

In [ ]:
# Configure fitting
config = GFAFitConfig(
    x_dim_init=10,  # Set to larger than the hypothesized latent dimensionality
    fit_tol=1e-8,  # Tolerance to determine fitting convergence
    max_iter=20000,  # Maximum number of fitting iterations
    verbose=True,  # Print fitting progress
    random_seed=0,  # Set to None for no seeding
    min_var_frac=0.001,  # Private variance floor
    prune_x=True,  # For speed-up, remove latents that become inactive
    prune_tol=1e-7,  # Tolerance for pruning inactive latents
    save_x=True,  # Set False to save memory when saving final results
    save_c_cov=True,  # Set False to save memory when saving final results
    save_fit_progress=True,  # Save lower bound, runtime each iteration
)

# Instantiate a GFA model with config
model = GFAModel(config=config)

# Fit the model (automatically initializes if needed)
model.fit(Y)

### Optional: Save the model to a file

In [ ]:
model.save("demo_model.safetensors")

### Optional: Load an existing model from a file

In [ ]:
model = GFAModel.load("demo_model.safetensors")

### Check fitting results

In [ ]:
# Display flags indicating fitting procedure status
model.flags.display()

# Plot the lower bound and cumulative runtime at each iteration
model.tracker.plot_lb()
model.tracker.plot_runtime()

### Visualize recovery of select parameters

#### Loading matrices

In [ ]:
# Ground truth
plt.figure(figsize=(3, 5))
plt.subplot(1, 2, 1)
plt.title("Ground truth C")
hinton_diagram(obs_params_true.C)

# Estimate
# NOTE: In general, the columns of the estimated loading matrices are unordered,
#       and will not match the order in the ground truth. Here, we can reorder
#       and flip the sign of each column to facilitate comparison.
reorder = np.array([1, 6, 4, 0, 3, 5, 2])
rescale = np.array([-1, -1, 1, 1, 1, -1, 1])

plt.subplot(1, 2, 2)
plt.title("Estimated C")
hinton_diagram(model.obs_posterior.C.mean[:, reorder] * rescale)

plt.tight_layout()
plt.show()

#### ARD parameters

In [ ]:
# Ground truth
# Compute the relative shared variance explained by each latent in each group
alpha_inv_true = 1 / obs_params_true.alpha
alpha_inv_rel_true = alpha_inv_true / np.sum(alpha_inv_true, axis=1, keepdims=True)
plt.figure(figsize=(5, 3))
plt.subplot(2, 1, 1)
plt.title("Ground truth ARD")
hinton_diagram(alpha_inv_rel_true)

# Estimate
# NOTE: In general, the columns of the estimated alpha are unordered, and will
#       not match the order in the ground truth. Here, we can reorder the
#       columns to facilitate comparison.

# Compute the relative shared variance explained by each latent in each group
alpha_inv_est = 1 / model.obs_posterior.alpha.mean
alpha_inv_rel_est = alpha_inv_est / np.sum(alpha_inv_est, axis=1, keepdims=True)
plt.subplot(2, 1, 2)
plt.title("Estimated ARD")
hinton_diagram(alpha_inv_rel_est[:, reorder])

plt.tight_layout()
plt.show()

### Inferring latents

In [ ]:
# For new data, use model.infer_latents(Y_new) to get latent posteriors.
X_new = model.infer_latents(Y)
print(f"Inferred latents shape: {X_new.mean.shape}")

## Explore the model fit with various descriptive statistics

### Performance metrics: Evidence lower bound and leave-group-out prediction

In [ ]:
# Evidence lower bound
lb = compute_lower_bound(
    Y, model.obs_posterior, model.latents_posterior, model.obs_hyperprior
)
print(f"Lower bound:        {lb:.4f}")

# Leave-group-out prediction
R2, MSE = gfa_stats.predictive_performance(Y, model.obs_posterior)
print(f"Leave-group-out R^2: {R2:.4f}")
print(f"                MSE: {MSE:.4f}")

# Leave-one-out prediction
R2, MSE = gfa_stats.predictive_performance(
    Y, model.obs_posterior, y_dims=np.ones(y_dims.sum(), dtype=int)
)
print(f"Leave-one-out R^2:   {R2:.4f}")
print(f"              MSE:   {MSE:.4f}")

### Estimated signal-to-noise ratios

In [ ]:
# Signal-to-noise ratios of each group, according to estimated model parameters
snr = model.obs_posterior.compute_snr()

# Format and print the snr values
print("Estimated SNRs:")
for group_idx in range(n_groups):
    print(f"    Group {group_idx + 1}: {snr[group_idx]:.4f}")

### Determine and then visualize the dimensionalities of all types

In [ ]:
# Compute dimensionalities and shared variance explained by each dimension type
(
    num_dim,
    sig_dims,
    var_exp,
    dim_types,
) = model.obs_posterior.compute_dimensionalities(
    cutoff_shared_var=0.02, cutoff_snr=0.001
)

# Visualize the number of each type of dimension
plt.figure(figsize=(4, 2))
plot_dimensionalities(
    num_dim, dim_types, group_names=["A", "B", "C"], plot_zero_dim=False
)
plt.show()

# Visualize the shared variance explained by each dimension type in each group
plt.figure(figsize=(4, 6))
plot_var_exp(var_exp, dim_types, group_names=["A", "B", "C"], plot_zero_dim=False)
plt.show()

### Perform a pairwise analysis of interactions between groups

In [ ]:
# Compute pairwise dimensionalities and shared variances explained
pair_dims, pair_var_exp, pairs = ObsParamsPosterior.compute_dims_pairs(
    num_dim, dim_types, var_exp
)

# Visualize pairwise dimensionalities
plt.figure(figsize=(7, 2.5))
plot_dims_pairs(pair_dims, pairs, n_groups, group_names=["A", "B", "C"])
plt.show()

# Visualize pairwise shared variances
plt.figure(figsize=(5, 2.5))
plot_var_exp_pairs(
    pair_var_exp, pairs, n_groups, group_names=["A", "B", "C"], sem_pair_var_exp=None
)